## Imports

In [ ]:

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "kaggle"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm", "torchstain", "transformers"], check=True)

import os, gc, glob, json, pathlib, zipfile, timeit, time, warnings
import numpy as np
import pandas as pd
import cv2
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from IPython.display import display, HTML, clear_output

warnings.filterwarnings('ignore', category=FutureWarning)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.backends import cudnn
import timm
import torchstain
from transformers import SegformerModel, SegformerConfig



## Optinal Persistence
We had to use google drive to persist our results due to google colab peridodically disconnecting a runtime or running out of compute units

In [ ]:

_DRIVE_AVAILABLE = False
try:
    from google.colab import drive as _gdrive
    _gdrive.mount('/content/drive', force_remount=False)
    _DRIVE_AVAILABLE = True
    print("✓ Google Drive mounted at /content/drive")
except Exception as _e:
    print(f"⚠ Google Drive not available ({_e}) — using /content/ (ephemeral).")
    print("  Checkpoints and datasets will be lost on session disconnect.")
    print("  To persist: run in Colab with Drive access.")

# Root persisted directory — everything lives here
if _DRIVE_AVAILABLE:
    _PERSIST_ROOT = '/content/drive/MyDrive/hubmap'
else:
    _PERSIST_ROOT = '/content/hubmap'

os.makedirs(_PERSIST_ROOT, exist_ok=True)
print(f"  Persist root: {_PERSIST_ROOT}")


## Kaggle Auth and Dataset Loading

In [ ]:
os.environ['KAGGLE_API_TOKEN'] = "Place Holder"

kaggle_dir = pathlib.Path.home() / '.kaggle'
kaggle_dir.mkdir(parents=True, exist_ok=True)

# Write access_token file (new token format)
_token_path = kaggle_dir / 'access_token'
_token_path.write_text(os.environ['KAGGLE_API_TOKEN'])
os.chmod(_token_path, 0o600)



def _free_disk_gb(path='/content') -> float:
    import shutil
    return shutil.disk_usage(path).free / 1_073_741_824


def _kaggle_download(cmd: list, max_retries: int = 6) -> subprocess.CompletedProcess:
    """
    Run a kaggle CLI download command with exponential backoff on 429 rate limits.
    Raises CalledProcessError on non-retryable failures.
    """
    wait = 60  # start at 60s — Kaggle rate limit window is ~1 minute
    for attempt in range(max_retries):
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode == 0:
            return result
        combined = (result.stdout + result.stderr).strip()
        if '429' in combined or 'Too Many Requests' in combined:
            if attempt < max_retries - 1:
                print(f"  ⏳ Kaggle rate-limited (429) — waiting {wait}s before retry "
                      f"({attempt+1}/{max_retries-1}) …")
                time.sleep(wait)
                wait = min(wait * 2, 300)  # cap at 5 minutes
                continue
        # Non-429 failure — raise immediately with full output
        raise subprocess.CalledProcessError(
            result.returncode, cmd, output=result.stdout, stderr=result.stderr
        )
    raise subprocess.CalledProcessError(max_retries, cmd, output='', stderr='Max retries exceeded')


def _download_single_file(slug: str, remote_path: str, local_dir: str):
    """Download one specific file from a Kaggle dataset by path."""
    try:
        _kaggle_download(
            ['kaggle', 'datasets', 'download', '-d', slug,
             '--file', remote_path, '-p', local_dir]
        )
    except subprocess.CalledProcessError:
        return None
    fname = os.path.basename(remote_path)
    for dp, _, fnames in os.walk(local_dir):
        if fname in fnames:
            return os.path.join(dp, fname)
    return None


def _list_kaggle_dataset_files(slug: str) -> list:
    """Return the list of file paths in a Kaggle dataset (uses kaggle CLI)."""
    result = subprocess.run(
        ['kaggle', 'datasets', 'files', '-d', slug, '--csv'],
        capture_output=True, text=True
    )
    lines = result.stdout.strip().splitlines()
    # CSV header: name,size,creationDate
    if len(lines) < 2:
        return []
    files = []
    for line in lines[1:]:
        parts = line.split(',')
        if parts:
            files.append(parts[0].strip())
    return files


def download_competition_data(competition: str, dest_dir: str):
    """Download + extract competition zip with retry/backoff and clear error output."""
    os.makedirs(dest_dir, exist_ok=True)
    print(f"  Downloading competition '{competition}' …")
    try:
        _kaggle_download(
            ['kaggle', 'competitions', 'download', '-c', competition, '-p', dest_dir]
        )
    except subprocess.CalledProcessError as e:
        _out = (e.stdout or '').strip()
        _err = (e.stderr or '').strip()
        _msg = _err or _out or '(no output — check Kaggle credentials)'
        raise RuntimeError(
            f"Competition download failed for '{competition}'.\n"
            f"  Kaggle says: {_msg[:500]}\n\n"
            f"  Common causes:\n"
            f"    • Token expired → update KAGGLE_API_TOKEN at top of script\n"
            f"    • Not accepted competition rules → go to kaggle.com/c/{competition}/rules\n"
            f"    • 429 rate limit → wait 60s and re-run (retry is automatic, 6 attempts)\n"
        ) from e
    zips = [os.path.join(dest_dir, f) for f in os.listdir(dest_dir) if f.endswith('.zip')]
    for zp in zips:
        print(f"  Extracting {os.path.basename(zp)} …")
        with zipfile.ZipFile(zp, 'r') as zf:
            for m in tqdm(zf.namelist(), desc="  Extracting", unit="file",
                          dynamic_ncols=True, leave=True):
                zf.extract(m, dest_dir)
        os.remove(zp)
    print(f"  Competition data ready.")


def fetch_lung_from_pseudo_csv(pseudo_dir: str, dest_dir: str,
                               competition_search_dirs: list | None = None,
                               n: int = 500) -> list:
    """
    Build a lung manifest using images + masks from the 3rd-place pseudo-label
    CSV (vladimirsydor/hubmap-2022-add-data-labels-v2) which we already download.

    Strategy:
    1. Parse all *lung*.csv files in pseudo_dir for (id, rle) rows.
    2. For each ID, look in competition_search_dirs + HUBMAP_IMG_DIR for the image.
    3. If the image is found on disk, add it to the manifest with its RLE mask.
    4. For IDs not found locally, selectively download from igorkrashenyi/lung-hpa-dataset
       using per-file Kaggle CLI calls (only N files, not the full 65k zip).
    5. Fallback: Zenodo Team_2.zip direct HTTP download (no Kaggle API needed).
    """
    import urllib.request

    local_dir = os.path.join(dest_dir, 'lung_hpa')
    os.makedirs(local_dir, exist_ok=True)
    done_f = os.path.join(local_dir, '.done')
    mf_f   = os.path.join(local_dir, 'manifest.json')

    if os.path.exists(done_f) and os.path.exists(mf_f):
        with open(mf_f) as f:
            cached = json.load(f)
        cached = [r for r in cached if os.path.exists(r['img_path'])]
        n_masked = sum(1 for r in cached if r.get('mask_path') or r.get('rle'))

        zenodo_done_check = os.path.join(local_dir, '.zenodo_done')
        lung_csvs_exist   = bool(glob.glob(
            os.path.join(pseudo_dir, '**', '*lung*.csv'), recursive=True))

        # Auto-invalidate ONLY if cache has 0 masks AND Zenodo hasn't been tried yet.
        # Once Zenodo ran, we keep the cache and instead patch RLEs retroactively.
        if n_masked == 0 and lung_csvs_exist and not os.path.exists(zenodo_done_check):
            print(f"  [CACHE INVALID] Lung cache has 0 masks, Zenodo not yet tried — "
                  f"rebuilding …")
            os.remove(done_f)
            os.remove(mf_f)
        else:
            # Retroactively patch any cached entries that are missing RLEs.
            # This runs fast (no downloads) and fixes the Zenodo images that were
            # extracted without being matched to hpa_lungs.csv masks.
            if n_masked < len(cached) and lung_csvs_exist:
                print(f"  [CACHE PATCH] {len(cached)-n_masked} entries lack RLE — "
                      f"matching against lung CSVs …")
                rle_lookup = {}
                for csv_path in glob.glob(
                        os.path.join(pseudo_dir, '**', '*lung*.csv'), recursive=True):
                    try:
                        _df = pd.read_csv(csv_path)
                        if 'encoding' in _df.columns and 'rle' not in _df.columns:
                            _df = _df.rename(columns={'encoding': 'rle'})
                        if 'id' in _df.columns and 'rle' in _df.columns:
                            for _, _r in _df.iterrows():
                                if pd.notna(_r.get('rle')) and \
                                        str(_r['rle']) not in ('', 'nan'):
                                    rle_lookup[str(_r['id'])] = str(_r['rle'])
                    except Exception:
                        pass
                print(f"    RLE lookup built: {len(rle_lookup)} lung IDs")
                patched = 0
                for r in cached:
                    if r.get('rle') or r.get('mask_path'):
                        continue  # already has a mask
                    stem = os.path.splitext(os.path.basename(r['img_path']))[0]
                    rle = rle_lookup.get(stem)
                    if rle is None:
                        suffix = stem.split('_')[-1]
                        for csv_id, csv_rle in rle_lookup.items():
                            if csv_id.endswith('_' + suffix):
                                rle = csv_rle
                                break
                    if rle:
                        r['rle'] = rle
                        patched += 1
                if patched:
                    with open(mf_f, 'w') as fout:
                        json.dump(cached, fout, indent=2)
                    print(f"    Patched {patched} entries with RLE — manifest updated on Drive")
                n_masked = sum(1 for r in cached if r.get('mask_path') or r.get('rle'))
            print(f"  [CACHED] Lung — {len(cached)} entries ({n_masked} with masks/RLE)")
            return cached

    # lung psuedo labels
    lung_csvs = glob.glob(os.path.join(pseudo_dir, '**', '*lung*.csv'), recursive=True)
    print(f"  Found {len(lung_csvs)} lung CSVs: {[os.path.basename(c) for c in lung_csvs]}")

    all_rows = []
    for csv_path in lung_csvs:
        try:
            df = pd.read_csv(csv_path)
            if 'encoding' in df.columns and 'rle' not in df.columns:
                df = df.rename(columns={'encoding': 'rle'})
            if 'id' in df.columns and 'rle' in df.columns:
                df = df[df['rle'].notna() & (df['rle'] != '')].copy()
                df['_source'] = os.path.basename(csv_path)
                all_rows.append(df[['id', 'rle', '_source']])
                print(f"    {os.path.basename(csv_path)}: {len(df)} masked lung rows")
        except Exception as e:
            print(f"    [WARN] Could not read {csv_path}: {e}")

    if not all_rows:
        print("  No lung pseudo-label CSVs found — lung data unavailable.")
        pathlib.Path(done_f).touch()
        with open(mf_f, 'w') as f:
            json.dump([], f)
        return []

    lung_df = pd.concat(all_rows, ignore_index=True).drop_duplicates('id')
    print(f"  Total unique lung IDs with pseudo-labels: {len(lung_df)}")

    # Cap to n rows
    if len(lung_df) > n:
        lung_df = lung_df.sample(n=n, random_state=42).reset_index(drop=True)
        print(f"  Sampled down to {n} rows")

    # which image IDs already exist on disk 
    search_dirs = list(competition_search_dirs or [])
    search_dirs.append(local_dir)

    disk_lookup = {}
    for sdir in search_dirs:
        if not os.path.isdir(sdir):
            continue
        for fname in os.listdir(sdir):
            stem = os.path.splitext(fname)[0]
            fpath = os.path.join(sdir, fname)
            if stem not in disk_lookup:
                disk_lookup[stem] = fpath

    manifest   = []
    need_dl    = []  # IDs not found on disk

    for _, row in lung_df.iterrows():
        img_id = str(row['id'])
        rle    = row['rle']
        found = disk_lookup.get(img_id)
        if not found:
            for sdir in search_dirs:
                for ext in ('.jpg', '.jpeg', '.png', '.tif', '.tiff'):
                    p = os.path.join(sdir, img_id + ext)
                    if os.path.exists(p):
                        found = p
                        break
                if found:
                    break
        if found:
            manifest.append({
                'img_path':    found,
                'mask_path':   None,  
                'rle':         rle,
                'organ':       'lung',
                'data_source': 'HPA',
                'pixel_size':  0.50,
            })
        else:
            need_dl.append((img_id, rle))

    print(f"  Found on disk: {len(manifest)} lung images")
    print(f"  Need download: {len(need_dl)} lung images")

    # The 2nd-place team (ConvNeXt-L, ~0.84 score) published their external data
    # on Zenodo at https://zenodo.org/records/7545745. Direct HTTP, no Kaggle API.
    ZENODO_URL  = 'https://zenodo.org/records/7545745/files/Team_2.zip'

    zenodo_done = os.path.join(local_dir, '.zenodo_done')
    zenodo_dir  = os.path.join(local_dir, 'zenodo_team2')

    if len(manifest) < 50 and not os.path.exists(zenodo_done):
        print(f"\n  ⚠ Only {len(manifest)} lung images found locally — "
              f"trying Zenodo Team_2.zip fallback (3.6 GB, no Kaggle API) …")
        os.makedirs(zenodo_dir, exist_ok=True)
        zenodo_zip = os.path.join(zenodo_dir, 'Team_2.zip')
        try:
            print(f"  Downloading {ZENODO_URL} …")
            # Stream in chunks so progress is visible
            req = urllib.request.urlopen(ZENODO_URL, timeout=60)
            total = int(req.headers.get('Content-Length', 0))
            chunk = 1024 * 1024  # 1 MB
            downloaded = 0
            with open(zenodo_zip, 'wb') as fout:
                while True:
                    data = req.read(chunk)
                    if not data:
                        break
                    fout.write(data)
                    downloaded += len(data)
                    if total:
                        pct = downloaded / total * 100
                        print(f"\r  {downloaded/1e6:.0f}/{total/1e6:.0f} MB  ({pct:.1f}%)",
                              end='', flush=True)
            print()
            # Extract lung-related files only
            print("  Extracting lung images from Team_2.zip …")
            lung_found = 0
            with zipfile.ZipFile(zenodo_zip, 'r') as zf:
                for member in zf.namelist():
                    if 'lung' in member.lower() and not member.endswith('/'):
                        zf.extract(member, zenodo_dir)
                        lung_found += 1
            os.remove(zenodo_zip)
            pathlib.Path(zenodo_done).touch()
            print(f"  Extracted {lung_found} lung-related files from Team_2.zip")
            # Build RLE lookup from all lung CSVs (stem → rle)
            rle_lookup = {}
            lung_csvs = glob.glob(
                os.path.join(pseudo_dir, '**', '*lung*.csv'), recursive=True)
            for csv_path in lung_csvs:
                try:
                    _df = pd.read_csv(csv_path)
                    if 'encoding' in _df.columns and 'rle' not in _df.columns:
                        _df = _df.rename(columns={'encoding': 'rle'})
                    if 'id' in _df.columns and 'rle' in _df.columns:
                        for _, _r in _df.iterrows():
                            if pd.notna(_r.get('rle')) and str(_r['rle']) not in ('', 'nan'):
                                rle_lookup[str(_r['id'])] = str(_r['rle'])
                except Exception:
                    pass
            print(f"  Built RLE lookup: {len(rle_lookup)} entries from lung CSVs")

            # Scan extracted lung images, match to RLEs by stem
            IMAGE_EXTS = {'.png', '.jpg', '.jpeg', '.tif', '.tiff'}
            n_rle_matched = 0
            for root, _, fnames in os.walk(zenodo_dir):
                for fname in fnames:
                    if os.path.splitext(fname)[1].lower() not in IMAGE_EXTS:
                        continue
                    if 'mask' in fname.lower():
                        continue
                    img_p  = os.path.join(root, fname)
                    stem   = os.path.splitext(fname)[0]
                    # Try exact match, then suffix match (e.g. "lung_6551" → "ENSG..._6551")
                    rle = rle_lookup.get(stem)
                    if rle is None:
                        suffix = stem.split('_')[-1]
                        for csv_id, csv_rle in rle_lookup.items():
                            if csv_id.endswith('_' + suffix):
                                rle = csv_rle
                                break
                    # Fallback: check for a sibling _mask.png file
                    mask_p = os.path.join(root, stem + '_mask.png')
                    has_mask_file = os.path.exists(mask_p)
                    if rle:
                        n_rle_matched += 1
                    manifest.append({
                        'img_path':    img_p,
                        'mask_path':   mask_p if has_mask_file else None,
                        'rle':         rle,
                        'organ':       'lung',
                        'data_source': 'HPA',
                        'pixel_size':  0.50,
                    })
            print(f"  RLE matched: {n_rle_matched}/{lung_found} Zenodo lung images")
        except Exception as e:
            print(f"  ⚠ Zenodo download failed: {e}")
            print(f"  Continuing with {len(manifest)} lung images found so far.")

    n_masked = sum(1 for r in manifest if r.get('mask_path') or r.get('rle'))
    with open(mf_f, 'w') as f:
        json.dump(manifest, f, indent=2)
    pathlib.Path(done_f).touch()
    print(f"\n  ✓ {len(manifest)} lung entries ready  ({n_masked} with masks/RLE)")
    return manifest


def fetch_spleen_from_pseudo_csvs(pseudo_dir: str, dest_dir: str,
                                   competition_search_dirs: list[str] | None = None,
                                   n: int = 500) -> list:
    """
    Build a spleen manifest from pseudo-label CSVs.

    Strategy (in order):
    1. Check competition train_images + test_images for any IDs that already
       exist on disk — these are free pseudo-labeled spleen images.
    2. For remaining GTEx IDs (gtx_spleen_v2/v3.csv, ~29-30 rows each), try
       to download from sakvaua/gtex-pseudo-humantorusteam. That dataset is
       confirmed to only hold kidney files, so this step is expected to yield
       0 currently; it is kept for forward-compatibility.

    Prints sample IDs from each CSV so the format is visible for debugging.
    """
    GTEX_SLUG = 'sakvaua/gtex-pseudo-humantorusteam'
    local_dir = os.path.join(dest_dir, 'gtex_spleen')
    os.makedirs(local_dir, exist_ok=True)
    done_f = os.path.join(local_dir, '.done')
    mf_f   = os.path.join(local_dir, 'manifest.json')

    if os.path.exists(done_f) and os.path.exists(mf_f):
        with open(mf_f) as f:
            cached = json.load(f)
        cached = [r for r in cached if os.path.exists(r['img_path'])]
        print(f"  [CACHED] GTEx/HPA spleen — {len(cached)} pairs")
        return cached

    spleen_csvs = glob.glob(os.path.join(pseudo_dir, '**', '*spleen*.csv'), recursive=True)
    print(f"  Found {len(spleen_csvs)} spleen CSVs: {[os.path.basename(c) for c in spleen_csvs]}")

    all_rows = []
    for csv_path in spleen_csvs:
        try:
            df = pd.read_csv(csv_path)
            if 'encoding' in df.columns and 'rle' not in df.columns:
                df = df.rename(columns={'encoding': 'rle'})
            if 'id' in df.columns and 'rle' in df.columns:
                df['_source_csv'] = os.path.basename(csv_path)
                all_rows.append(df)
                sample_ids = df['id'].astype(str).head(3).tolist()
                print(f"    {os.path.basename(csv_path)}: {len(df)} rows  "
                      f"sample IDs: {sample_ids}")
        except Exception as e:
            print(f"    [SKIP] {os.path.basename(csv_path)}: {e}")

    if not all_rows:
        print(f"  [WARN] No usable spleen CSVs found")
        return []

    spleen_df = pd.concat(all_rows, ignore_index=True).drop_duplicates(subset='id')
    print(f"  Total unique spleen IDs: {len(spleen_df)}")

    IMAGE_EXTS = {'.tiff', '.tif', '.png', '.jpg', '.jpeg'}
    manifest   = []
    used_ids   = set()

    print(f"  Building image index over pseudo_labels dir …")
    pseudo_img_index = _build_image_index(pseudo_dir)
    print(f"  Pseudo-labels dir: {len(pseudo_img_index)} image files found")

    for _, row in spleen_df.iterrows():
        img_id   = str(row['id'])
        rle_val  = str(row.get('rle', '')) if pd.notna(row.get('rle')) else ''
        pixel_sz = float(row['pixel_size']) if pd.notna(row.get('pixel_size')) else 0.49

        stem = img_id.lower()
        if stem not in pseudo_img_index:
            continue

        entry = {
            'img_path':    pseudo_img_index[stem],
            'mask_path':   None,
            'organ':       'spleen',
            'data_source': str(row.get('data_source', 'GTEx')),
            'pixel_size':  pixel_sz,
        }
        if rle_val and rle_val != 'nan':
            entry['rle']     = rle_val
            entry['use_rle'] = True
        manifest.append(entry)
        used_ids.add(img_id)

    print(f"  Found {len(manifest)} spleen images inside pseudo-labels dir")

    search_dirs = list(competition_search_dirs or [])
    if search_dirs:
        print(f"  Checking {len(search_dirs)} competition image dir(s) for spleen ID matches …")
        on_disk_count = 0
        for _, row in spleen_df.iterrows():
            img_id  = str(row['id'])
            if img_id in used_ids:
                continue
            rle_val = str(row.get('rle', '')) if pd.notna(row.get('rle')) else ''
            pixel_sz = float(row['pixel_size']) if pd.notna(row.get('pixel_size')) else 0.49

            img_path = None
            for d in search_dirs:
                for ext in IMAGE_EXTS:
                    candidate = os.path.join(d, f"{img_id}{ext}")
                    if os.path.exists(candidate):
                        img_path = candidate
                        break
                if img_path:
                    break

            if img_path is None:
                continue

            entry = {
                'img_path':    img_path,
                'mask_path':   None,
                'organ':       'spleen',
                'data_source': str(row.get('data_source', 'HPA')),
                'pixel_size':  pixel_sz,
            }
            if rle_val and rle_val != 'nan':
                entry['rle']     = rle_val
                entry['use_rle'] = True

            manifest.append(entry)
            used_ids.add(img_id)
            on_disk_count += 1

        print(f"  Found {on_disk_count} spleen images already on disk")

    gtex_df = spleen_df[
        spleen_df['_source_csv'].str.startswith('gtx_') &
        ~spleen_df['id'].astype(str).isin(used_ids)
    ].copy() if '_source_csv' in spleen_df.columns else pd.DataFrame()

    if not gtex_df.empty:
        print(f"  Attempting GTEx download for {len(gtex_df)} remaining GTEx-sourced IDs …")

        remote_files = _list_kaggle_dataset_files(GTEX_SLUG)
        remote_img_by_stem = {}
        for rf in remote_files:
            if os.path.splitext(rf)[1].lower() in IMAGE_EXTS:
                stem = os.path.splitext(os.path.basename(rf))[0].lower()
                remote_img_by_stem[stem] = rf

        print(f"  GTEx remote stems (first 5): {list(remote_img_by_stem.keys())[:5]}")

        dl_ok = dl_fail = 0
        for _, row in gtex_df.iterrows():
            img_id   = str(row['id'])
            id_lower = img_id.lower()
            rle_val  = str(row.get('rle', '')) if pd.notna(row.get('rle')) else ''
            pixel_sz = float(row['pixel_size']) if pd.notna(row.get('pixel_size')) else 0.49

            remote_path = None
            for stem_cand in [id_lower, f'spleen_{id_lower}']:
                if stem_cand in remote_img_by_stem:
                    remote_path = remote_img_by_stem[stem_cand]
                    break
            if remote_path is None:
                for stem, rf in remote_img_by_stem.items():
                    if id_lower in stem or stem in id_lower:
                        remote_path = rf
                        break

            if remote_path is None:
                dl_fail += 1
                continue

            result = _download_single_file(GTEX_SLUG, remote_path, local_dir)
            if result is None or not os.path.exists(result):
                dl_fail += 1
                continue

            entry = {
                'img_path':    result,
                'mask_path':   None,
                'organ':       'spleen',
                'data_source': 'GTEx',
                'pixel_size':  pixel_sz,
            }
            if rle_val and rle_val != 'nan':
                entry['rle']     = rle_val
                entry['use_rle'] = True
            manifest.append(entry)
            dl_ok += 1

        print(f"  GTEx download: {dl_ok} found, {dl_fail} not in dataset "
              f"(expected — dataset confirmed kidney-only)")

    print(f"  ✓ Total spleen manifest: {len(manifest)} entries")

    with open(mf_f, 'w') as f:
        json.dump(manifest, f, indent=2)
    pathlib.Path(done_f).touch()
    return manifest


def fetch_all_organ_pseudo_labels(pseudo_dir: str, hubmap_img_dir: str,
                                   extra_search_dirs: list[str] | None = None) -> list:
    """
    Match pseudo-label CSV rows against:
      1. Images inside pseudo_dir itself (most likely source — the 5.18 GB dataset
         probably contains actual image files for each CSV row)
      2. Competition train_images and any extra directories (e.g. test_images)

    Returns entries for all rows where the image file already exists on disk.
    """
    all_csvs = glob.glob(os.path.join(pseudo_dir, '**', '*.csv'), recursive=True)
    manifest = []

    ORGAN_NORM = {
        'colon': 'largeintestine', 'large_intestine': 'largeintestine',
        'prostate': 'prostate', 'kidney': 'kidney',
        'lung': 'lung', 'spleen': 'spleen',
    }


    print(f"  Building image index over pseudo_labels dir …")
    pseudo_img_index = _build_image_index(pseudo_dir)
    print(f"  Pseudo-labels dir contains {len(pseudo_img_index)} image files")

    flat_search_dirs = [hubmap_img_dir]
    for d in (extra_search_dirs or []):
        if d and os.path.isdir(d) and d not in flat_search_dirs:
            flat_search_dirs.append(d)

    def _find_image(img_id: str) -> str | None:
        stem = img_id.lower()
        if stem in pseudo_img_index:
            return pseudo_img_index[stem]
        for search_dir in flat_search_dirs:
            for ext in ('.tiff', '.tif', '.png', '.jpg'):
                candidate = os.path.join(search_dir, f"{img_id}{ext}")
                if os.path.exists(candidate):
                    return candidate
        return None

    seen_ids: set[str] = set()

    all_known_stems = set(pseudo_img_index.keys())
    for sd in flat_search_dirs:
        if os.path.isdir(sd):
            for fn in os.listdir(sd):
                all_known_stems.add(os.path.splitext(fn)[0].lower())

    _SKIP_CSVS = {'hpa_lungs.csv', 'hpa_spleen.csv'}

    for csv_path in all_csvs:
        if os.path.basename(csv_path) in _SKIP_CSVS:
            continue 
        try:
            df = pd.read_csv(csv_path)
            if 'encoding' in df.columns and 'rle' not in df.columns:
                df = df.rename(columns={'encoding': 'rle'})
            if 'id' not in df.columns or 'rle' not in df.columns:
                continue
        except Exception:
            continue


        df = df[df['rle'].notna() & (df['rle'].astype(str) != 'nan')].copy()
        df['_id_lower'] = df['id'].astype(str).str.lower()
        df = df[df['_id_lower'].isin(all_known_stems)]
        df = df[~df['id'].astype(str).isin(seen_ids)]
        if df.empty:
            continue

        csv_matched = 0
        for _, row in df.iterrows():
            img_id  = str(row['id'])
            rle     = str(row['rle'])
            img_path = _find_image(img_id)
            if img_path is None:
                continue

            organ_raw = str(row.get('organ', '')).lower().strip()
            organ     = ORGAN_NORM.get(organ_raw, organ_raw)
            if organ not in ('kidney', 'prostate', 'largeintestine', 'spleen', 'lung'):
                continue

            manifest.append({
                'id':          img_id,
                'img_path':    img_path,
                'mask_path':   None,
                'rle':         rle,
                'organ':       organ,
                'data_source': str(row.get('data_source', 'HPA')),
                'pixel_size':  float(row['pixel_size']) if pd.notna(row.get('pixel_size')) else 0.40,
                'use_rle':     True,
            })
            seen_ids.add(img_id)
            csv_matched += 1

        if csv_matched:
            print(f"    {os.path.basename(csv_path)}: {csv_matched} matches")

    if manifest:
        organs = pd.DataFrame(manifest)['organ'].value_counts().to_dict()
        print(f"  ✓ {len(manifest)} pseudo-label matches — {organs}")
    else:
        print(f"  (No pseudo-label IDs matched any image source)")
    return manifest


def _build_image_index(directory: str, recursive: bool = True) -> dict:
    """
    Walk directory and return {stem_lowercase: absolute_path} for every image file.
    Used to quickly match pseudo-label IDs to images stored anywhere in a large dir tree.
    """
    IMAGE_EXTS = {'.tiff', '.tif', '.png', '.jpg', '.jpeg'}
    index = {}
    walker = os.walk(directory) if recursive else [(directory, [], os.listdir(directory))]
    for dp, _, fnames in walker:
        for fn in fnames:
            if os.path.splitext(fn)[1].lower() in IMAGE_EXTS:
                stem = os.path.splitext(fn)[0].lower()
                index[stem] = os.path.join(dp, fn)
    return index


def fetch_hpa_images_for_pseudo_labels(pseudo_dir: str, data_dir: str) -> list[str]:
    """
    Return search directories for HPA images referenced by pseudo-label CSVs.

    The competition already downloads both train_images and test_images when you
    run download_competition_data(). The HPA images in the competition are stored
    there under their competition IDs — no separate Kaggle download is needed.

    Returns a list of directories to search (in priority order).
    """
    test_img_dir  = os.path.join(data_dir, 'test_images')
    train_img_dir = os.path.join(data_dir, 'train_images')

    found_dirs = []
    for d in (test_img_dir, train_img_dir):
        if os.path.isdir(d):
            n = sum(1 for f in os.listdir(d) if f.endswith(('.tiff', '.tif', '.png')))
            print(f"  Found competition images dir: {d}  ({n} images)")
            found_dirs.append(d)

    if not found_dirs:
        print("  [WARN] No competition image directories found — HPA images unavailable.")
    return found_dirs


# Download the data

DATA_DIR   = os.path.join(_PERSIST_ROOT, 'data')
EXT_DIR    = os.path.join(_PERSIST_ROOT, 'data', 'external')
PSEUDO_DIR = os.path.join(_PERSIST_ROOT, 'data', 'external', 'pseudo_labels')
MODELS_DIR = os.path.join(_PERSIST_ROOT, 'models')
os.makedirs(DATA_DIR,   exist_ok=True)
os.makedirs(EXT_DIR,    exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
print(f"  DATA_DIR   → {DATA_DIR}")
print(f"  MODELS_DIR → {MODELS_DIR}")

# Competition data
if not os.path.exists(os.path.join(DATA_DIR, 'train.csv')):
    print("Downloading HuBMAP competition data …")
    download_competition_data('hubmap-organ-segmentation', DATA_DIR)
else:
    print("HuBMAP data already present — skipping.")

HUBMAP_IMG_DIR = os.path.join(DATA_DIR, 'train_images')

# Integrity check
_n_train_tiffs = len(glob.glob(os.path.join(HUBMAP_IMG_DIR, '*.tiff')))
if _n_train_tiffs < 300 and os.path.exists(os.path.join(DATA_DIR, 'train.csv')):
    print(f"⚠ Only {_n_train_tiffs} train TIFFs found (expected ~351) — "
          f"Drive was likely full during download. Re-downloading competition data …")
    import shutil as _sh
    _sh.rmtree(DATA_DIR, ignore_errors=True)
    os.makedirs(DATA_DIR, exist_ok=True)
    download_competition_data('hubmap-organ-segmentation', DATA_DIR)
elif _n_train_tiffs < 351:
    print(f"⚠ {_n_train_tiffs}/351 train TIFFs on Drive — "
          f"{351 - _n_train_tiffs} may be corrupted. Training will skip missing files.")

PSEUDO_SLUG = 'vladimirsydor/hubmap-2022-add-data-labels-v2'
pseudo_done = os.path.join(PSEUDO_DIR, '.done')

# clear from data if corrupted
if os.path.exists(pseudo_done):
    if pathlib.Path(pseudo_done).read_text().strip() == 'unavailable':
        os.remove(pseudo_done)
        print("\nCleared stale pseudo-label sentinel — will retry download.")
    else:
        _n_csvs = len(glob.glob(os.path.join(PSEUDO_DIR, '**', '*.csv'), recursive=True))
        if _n_csvs < 10:
            os.remove(pseudo_done)
            print(f"\n⚠ Pseudo-label data corrupted ({_n_csvs} CSVs found, expected 20+) "
                  f"— forcing re-download.")

if not os.path.exists(pseudo_done):
    print("\nDownloading pseudo-labels (CSVs + masks only) …")
    os.makedirs(PSEUDO_DIR, exist_ok=True)
    try:
        _kaggle_download(
            ['kaggle', 'datasets', 'download', '-d', PSEUDO_SLUG, '-p', PSEUDO_DIR]
        )
        zips = [os.path.join(PSEUDO_DIR, f) for f in os.listdir(PSEUDO_DIR) if f.endswith('.zip')]
        for zp in zips:
            with zipfile.ZipFile(zp, 'r') as zf:
                for m in tqdm(zf.namelist(), desc="  Extracting pseudo-labels",
                              unit="file", dynamic_ncols=True, leave=True):
                    zf.extract(m, PSEUDO_DIR)
            os.remove(zp)
        pathlib.Path(pseudo_done).touch()
        print("  Pseudo-labels ready.")
    except subprocess.CalledProcessError as e:
        _out = (e.stdout or '').strip()
        _err = (e.stderr or '').strip()
        _msg = _err or _out or '(no output from kaggle CLI)'
        print(f"  ⚠ Pseudo-label download failed.")
        print(f"    Kaggle says: {_msg[:300]}")
        print(f"    Continuing without external pseudo-labels — HuBMAP data only.")
       
else:
    print("\nPseudo-labels already present — skipping.")


print("\n── Locating HPA images (competition test_images + train_images) ──")
COMPETITION_IMG_DIRS = fetch_hpa_images_for_pseudo_labels(PSEUDO_DIR, DATA_DIR)

print("\n── Checking pseudo-labels against competition images ──")
pseudo_hubmap_rows = fetch_all_organ_pseudo_labels(
    PSEUDO_DIR, HUBMAP_IMG_DIR,
    extra_search_dirs=COMPETITION_IMG_DIRS,
)


print("\n── Lung external data ──")
lung_manifest = fetch_lung_from_pseudo_csv(
    PSEUDO_DIR, EXT_DIR,
    competition_search_dirs=COMPETITION_IMG_DIRS,
    n=500,
)
n_lung_masked = sum(1 for r in lung_manifest if r.get('mask_path') or r.get('rle'))
print(f"  Lung mask/RLE check: {n_lung_masked}/{len(lung_manifest)} images have masks")

print("\n── Spleen external data ──")
spleen_manifest = fetch_spleen_from_pseudo_csvs(
    PSEUDO_DIR, EXT_DIR,
    competition_search_dirs=COMPETITION_IMG_DIRS,
    n=500,
)

print(f"\n── External data summary ──")
print(f"  Pseudo-label matches in image dirs: {len(pseudo_hubmap_rows)}")
print(f"  Lung HPA   : {len(lung_manifest)} entries  ({n_lung_masked} with masks)")
print(f"  Spleen GTEx: {len(spleen_manifest)} pairs")
total_ext = len(pseudo_hubmap_rows) + len(lung_manifest) + len(spleen_manifest)
print(f"  Total external: {total_ext}")


_ckpt_best  = os.path.join(MODELS_DIR, 'multimodal_unet_best.pth')
_ckpt_info  = os.path.join(MODELS_DIR, 'best_model_info.json')
print(f"\n── Checkpoint recovery status ──")
if os.path.exists(_ckpt_best):
    _mtime = time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(_ckpt_best)))
    _sz_mb = os.path.getsize(_ckpt_best) / 1e6
    if os.path.exists(_ckpt_info):
        with open(_ckpt_info) as _f:
            _ci = json.load(_f)
        print(f"  ✓ Found saved checkpoint: epoch {_ci['epoch']},  "
              f"val_loss {_ci['val_loss']:.4f},  saved {_mtime}")
        print(f"    Organ Dice: { {k: f'{v:.3f}' for k,v in _ci.get('organ_dice',{}).items()} }")
    else:
        print(f"  ✓ Found checkpoint ({_sz_mb:.0f} MB, saved {_mtime}) — no sidecar info")
    print(f"  → Training will resume from scratch but the best checkpoint is preserved.")
    print(f"    To skip training and run inference only, set SKIP_TRAINING=True below.")
else:
    print(f"  ✗ No checkpoint found — will train from scratch.")

SKIP_TRAINING    = False  # ← set True to jump straight to inference using saved checkpoint
RESUME_TRAINING  = False  # ← set True to load best checkpoint and continue training from that epoch



## Config

In [ ]:

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:64"
cudnn.benchmark = True
t0 = timeit.default_timer()

SUBMISSION_FILE = 'submission.csv'
TEST_BATCH_SIZE = 1
IMG_SIZE        = (768, 768)

ORGAN_MAP  = {'kidney': 0, 'prostate': 1, 'largeintestine': 2, 'spleen': 3, 'lung': 4}
SOURCE_MAP = {'Hubmap': 0, 'HPA': 1, 'GTEx': 1}


ORGAN_THRESHOLDS = {
    'Hubmap': {'kidney': 90,  'prostate': 100, 'largeintestine': 80,  'spleen': 100, 'lung': 15},
    'HPA':    {'kidney': 127, 'prostate': 127, 'largeintestine': 127, 'spleen': 127, 'lung': 25},
    'GTEx':   {'kidney': 127, 'prostate': 127, 'largeintestine': 127, 'spleen': 127, 'lung': 25},
}

ORGAN_PIXEL_SIZE_DEFAULTS = {
    'kidney': 0.50, 'prostate': 0.27, 'largeintestine': 0.23,
    'spleen': 0.49, 'lung': 0.50, 'unknown': 0.40,
}

USE_SEGFORMER     = False
SEGFORMER_VARIANT = 'nvidia/mit-b4' 

if not torch.cuda.is_available():
    raise RuntimeError(
        "\n\n  ✗ No GPU detected.\n"
        "  Runtime → Change runtime type → GPU (T4) → Disconnect & reconnect.\n"
    )
print(f"GPU: {torch.cuda.get_device_name(0)}  "
      f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")



## Stain Normalization

In [ ]:

_STAIN_NORMALIZER = None
_STAIN_TARGET_PATH: str | None = None


def _build_stain_normalizer(hubmap_img_dir: str) -> None:
    """
    Load the first available HuBMAP training image and fit a Macenko normalizer.
    Called once; result cached in _STAIN_NORMALIZER.
    """
    global _STAIN_NORMALIZER, _STAIN_TARGET_PATH
    if _STAIN_NORMALIZER is not None:
        return

    imgs = sorted(glob.glob(os.path.join(hubmap_img_dir, '*.tiff')))
    if not imgs:
        print("[WARN] No HuBMAP TIFF found — stain normalization disabled.")
        return

    ref_path = imgs[0]
    ref_bgr  = cv2.imread(ref_path, cv2.IMREAD_COLOR)
    if ref_bgr is None:
        print(f"[WARN] Could not read reference image {ref_path} — stain normalization disabled.")
        return

    ref_rgb = cv2.cvtColor(ref_bgr, cv2.COLOR_BGR2RGB)
    ref_t   = torch.from_numpy(ref_rgb).permute(2, 0, 1).float()

    try:
        normalizer = torchstain.normalizers.MacenkoNormalizer(backend='torch')
        normalizer.fit(ref_t)
        _STAIN_NORMALIZER  = normalizer
        _STAIN_TARGET_PATH = ref_path
        print(f"  Stain normalizer fitted on {os.path.basename(ref_path)}")
    except Exception as e:
        print(f"[WARN] Macenko fit failed ({e}) — stain normalization disabled.")


def _normalize_stain(img_bgr: np.ndarray) -> np.ndarray:
    """
    Normalize a BGR uint8 image to the reference H&E stain.
    Returns a BGR uint8 image. Falls back to input on any failure.

    Handles both torchstain API variants:
      - stains=True  → returns (I_norm, H, E) tuple
      - stains=False → returns I_norm tensor directly (some versions)
    The unpack `normed, _, _ = tensor` would silently iterate over dim-0
    giving a 2D slice, so we always call with stains=True and index result[0].
    """
    if _STAIN_NORMALIZER is None:
        return img_bgr
    try:
        rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        t   = torch.from_numpy(rgb).permute(2, 0, 1).float()
        # stains=True always returns a tuple (I_norm, H, E) across all versions
        result = _STAIN_NORMALIZER.normalize(I=t, stains=True)
        normed = result[0] if isinstance(result, (tuple, list)) else result
        if not isinstance(normed, torch.Tensor) or normed.ndim != 3 or normed.shape[0] != 3:
            return img_bgr
        normed_np = normed.permute(1, 2, 0).clamp(0, 255).byte().numpy()
        if normed_np.shape != img_bgr.shape:
            return img_bgr
        return cv2.cvtColor(normed_np, cv2.COLOR_RGB2BGR)
    except Exception:
        return img_bgr




## Utils and Datasets

In [ ]:


def rle_encode_less_memory(img):
    pixels = img.T.flatten()
    pixels[0] = 0; pixels[-1] = 0
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 2
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)


def rle_decode(mask_rle, shape):
    s = mask_rle.split()
    starts, lengths = [np.asarray(x, dtype=int) for x in (s[0:][::2], s[1:][::2])]
    starts -= 1
    ends = starts + lengths
    img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1
    return img.reshape((shape[1], shape[0])).T


def preprocess_inputs(x):
    x = np.asarray(x, dtype='float32')
    x /= 127.0; x -= 1.0
    return x


class HubmapTestDataset(Dataset):
    def __init__(self, df, data_dir, target_size):
        self.df = df; self.data_dir = data_dir; self.target_size = target_size

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        img0 = cv2.imread(os.path.join(self.data_dir, f"{row['id']}.tiff"),
                          cv2.IMREAD_UNCHANGED)
        if img0 is None:
            raise FileNotFoundError(f"{row['id']}.tiff not found in {self.data_dir}")
        img0 = CombinedTrainDataset._ensure_3channel(img0)
        orig_h, orig_w = img0.shape[:2]
        img  = cv2.resize(img0, self.target_size)
        img  = preprocess_inputs(img)
        img_t = torch.from_numpy(img.transpose((2, 0, 1)).copy()).float()
        return {
            'id': row['id'], 'img': img_t,
            'organ_name':  row['organ'],
            'source_name': row['data_source'],
            'organ_idx':   ORGAN_MAP.get(row['organ'], 5),
            'source_idx':  SOURCE_MAP.get(row['data_source'], 2),
            'pixel_size':  row.get('pixel_size', 0.4),
            'orig_h': orig_h, 'orig_w': orig_w,
        }


def build_external_df(lung_manifest: list, spleen_manifest: list,
                       pseudo_hubmap_rows: list) -> pd.DataFrame:
    """
    Combine all external manifests and compute has_real_mask per row.
    Rows without a real supervision signal must never appear in validation.
    """
    rows = lung_manifest + spleen_manifest + pseudo_hubmap_rows
    if not rows:
        print("[WARN] No external data — training on HuBMAP only.")
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    df = df[df['img_path'].apply(os.path.exists)].reset_index(drop=True)

    def _has_real_mask(row):
        # RLE-based mask — either explicit use_rle=True flag OR any non-null rle string
        rle_val = row.get('rle')
        if rle_val and pd.notna(rle_val) and str(rle_val) not in ('nan', ''):
            return True
        # PNG mask on disk
        mp = row.get('mask_path')
        if mp and pd.notna(mp) and os.path.exists(str(mp)):
            return True
        return False

    df['has_real_mask'] = df.apply(_has_real_mask, axis=1)

    counts   = df['organ'].value_counts().to_dict()
    n_masked = int(df['has_real_mask'].sum())
    print(f"[INFO] External df: {len(df)} rows — {counts}")
    print(f"[INFO]   With real masks (usable for val): {n_masked}")
    print(f"[INFO]   Train-only (no mask):             {len(df) - n_masked}")
    return df


class CombinedTrainDataset(Dataset):
    """
    Handles three row types:
      1. HuBMAP  — id + rle columns, loads TIFF from hubmap_img_dir
      2. HPA/GTEx with PNG mask — img_path + mask_path columns
      3. Pseudo-label — img_path (TIFF) + rle column, use_rle=True
    HPA/GTEx images are stain-normalized to the H&E reference before augmentation.
    """
    HEAVY_AUG_ORGANS = {'lung', 'spleen'}

    def __init__(self, df, hubmap_img_dir, target_size, is_train=True):
        self.df          = df.reset_index(drop=True)
        self.hubmap_dir  = hubmap_img_dir
        self.target_size = target_size
        self.is_train    = is_train

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img, mask = self._load(row)
        except Exception as e:
            print(f"[ERROR] idx={idx}: {e}")
            img  = np.zeros((*self.target_size, 3), dtype=np.uint8)
            mask = np.zeros(self.target_size, dtype=np.uint8)

        # ── Pixel-scale normalisation ────────────────────────────────────────
        # Cap the image to 2× target size BEFORE pixel-scale rescaling so we
        # never create a 10000×10000 intermediate in a DataLoader worker.
        # This alone brings per-sample time from ~3s → ~0.2s on large TIFFs.
        MAX_PRE = self.target_size[0] * 2   # e.g. 1536 for target=768
        if img.shape[0] > MAX_PRE or img.shape[1] > MAX_PRE:
            img  = cv2.resize(img,  (MAX_PRE, MAX_PRE))
            mask = cv2.resize(mask, (MAX_PRE, MAX_PRE),
                              interpolation=cv2.INTER_NEAREST)

        REF_PIXEL_SIZE = 0.4  # µm/pixel — HPA canonical resolution
        try:
            pixel_size = float(row.get('pixel_size', REF_PIXEL_SIZE))
        except (TypeError, ValueError):
            pixel_size = REF_PIXEL_SIZE
        if pixel_size > 0:
            scale = np.clip(pixel_size / REF_PIXEL_SIZE, 0.25, 4.0)
            if abs(scale - 1.0) > 0.05:
                new_h = max(64, int(img.shape[0] * scale))
                new_w = max(64, int(img.shape[1] * scale))
                img  = cv2.resize(img,  (new_w, new_h))
                mask = cv2.resize(mask, (new_w, new_h),
                                  interpolation=cv2.INTER_NEAREST)

        # Final resize to training resolution
        img  = cv2.resize(img,  self.target_size)
        mask = cv2.resize(mask, self.target_size, interpolation=cv2.INTER_NEAREST)

        # Stain normalization after resize (cheap at 768×768)
        img = self._maybe_normalize_stain(img, row)

        if self.is_train:
            img, mask = self._augment(img, mask, row['organ'])

        # Hard shape guard — ensures every item in a batch has identical tensor
        # size before collation. Catches any path that produces wrong shapes.
        th, tw = self.target_size[1], self.target_size[0]
        if img.shape != (th, tw, 3):
            img = self._ensure_3channel(img)
            if img.shape[:2] != (th, tw):
                img = cv2.resize(img, (tw, th))

        return self._to_tensors(img, mask, row)

    @staticmethod
    def _ensure_3channel(img: np.ndarray) -> np.ndarray:
        """Guarantee image is (H, W, 3) BGR regardless of source format."""
        if img.ndim == 2:
            return cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
        if img.ndim == 3:
            c = img.shape[2]
            if c == 1:
                return cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
            if c == 4:
                return cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)
            if c == 3:
                return img
        # Unexpected shape — take first channel and replicate to BGR
        flat = img.reshape(img.shape[0], img.shape[1], -1)[:, :, :1]
        return np.repeat(flat, 3, axis=2).astype(np.uint8)

    def _load(self, row):
        has_img_path = pd.notna(row.get('img_path')) and bool(row.get('img_path'))
        # use_rle: either explicit flag, or any non-null rle value on an img_path row
        rle_val = row.get('rle')
        has_rle = bool(rle_val) and pd.notna(rle_val) and str(rle_val) not in ('nan', '')
        use_rle = bool(row.get('use_rle', False)) or (has_img_path and has_rle)

        if has_img_path and use_rle:
            # Pseudo-label or lung HPA row: image file, mask from RLE string
            img = cv2.imread(row['img_path'], cv2.IMREAD_UNCHANGED)
            if img is None:
                raise FileNotFoundError(row['img_path'])
            img = self._ensure_3channel(img)
            # Stain normalization is deferred to __getitem__ (runs after resize)

            rle_str = str(row.get('rle', ''))
            if not rle_str or rle_str == 'nan':
                mask = np.zeros(img.shape[:2], dtype=np.uint8)
            else:
                mask = rle_decode(rle_str, img.shape[:2])
            return img, mask

        elif has_img_path:
            # HPA/GTEx row: image file + optional PNG mask
            img = cv2.imread(row['img_path'], cv2.IMREAD_UNCHANGED)
            if img is None:
                raise FileNotFoundError(row['img_path'])
            img = self._ensure_3channel(img)
            # Stain normalization is deferred to __getitem__ (runs after resize)

            mask_path = row.get('mask_path')
            if mask_path and pd.notna(mask_path) and os.path.exists(str(mask_path)):
                raw = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)
                if raw is None:
                    mask = np.zeros(img.shape[:2], dtype=np.uint8)
                else:
                    if raw.ndim != 2:
                        raw = cv2.cvtColor(raw, cv2.COLOR_BGR2GRAY)
                    source   = str(row.get('data_source', 'HPA'))
                    organ_nm = str(row.get('organ', 'kidney'))
                    thresh_v = ORGAN_THRESHOLDS.get(source, ORGAN_THRESHOLDS['HPA']).get(organ_nm, 127)
                    mask = (raw > thresh_v).astype(np.uint8)
            else:
                mask = np.zeros(img.shape[:2], dtype=np.uint8)
            return img, mask

        else:
            # HuBMAP row: TIFF + RLE
            path = os.path.join(self.hubmap_dir, f"{row['id']}.tiff")
            img  = cv2.imread(path, cv2.IMREAD_UNCHANGED)
            if img is None:
                raise FileNotFoundError(path)
            img  = self._ensure_3channel(img)
            mask = rle_decode(row['rle'], img.shape[:2])
            return img, mask

    @staticmethod
    def _maybe_normalize_stain(img: np.ndarray, row) -> np.ndarray:
        """
        Apply Macenko stain normalization to HuBMAP images only.

        Rationale from 3rd place solution analysis: HPA images come from a single
        standardised protocol (same scanner, same lab) so their stain is already
        consistent. HuBMAP images vary across donors and labs, so normalising them
        towards the HPA reference reduces the domain gap the model must overcome.
        """
        source = str(row.get('data_source', 'HPA'))
        if source.lower() == 'hubmap' and _STAIN_NORMALIZER is not None:
            return _normalize_stain(img)
        return img

    def _augment(self, img, mask, organ):
        if np.random.rand() > 0.5:
            img, mask = np.fliplr(img).copy(), np.fliplr(mask).copy()
        if np.random.rand() > 0.5:
            img, mask = np.flipud(img).copy(), np.flipud(mask).copy()
        if organ in self.HEAVY_AUG_ORGANS:
            k = np.random.choice([0, 1, 2, 3])
            if k:
                img  = np.rot90(img,  k=k).copy()
                mask = np.rot90(mask, k=k).copy()
            alpha = 1.0 + np.random.uniform(-0.15, 0.15)
            beta  = np.random.uniform(-20, 20)
            img   = np.clip(img.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
            if np.random.rand() > 0.6:
                ksize = np.random.choice([3, 5])
                img   = cv2.GaussianBlur(img, (ksize, ksize), 0)
        return img, mask

    def _to_tensors(self, img, mask, row):
        organ_name  = row['organ']
        source_name = str(row.get('data_source', 'HPA'))
        pixel_size  = float(row.get('pixel_size',
                            ORGAN_PIXEL_SIZE_DEFAULTS.get(organ_name, 0.4)))
        img_t  = torch.from_numpy(
            preprocess_inputs(img).transpose((2, 0, 1)).copy()).float()
        mask_t = torch.from_numpy(mask.copy()).float().unsqueeze(0)
        return {
            'img': img_t, 'mask': mask_t,
            'organ_name':  organ_name,
            'source_name': source_name,
            'organ_idx':   ORGAN_MAP.get(organ_name, 5),
            'source_idx':  SOURCE_MAP.get(source_name, 1),
            'pixel_scale': pixel_size,
        }


def make_weighted_sampler(df: pd.DataFrame) -> WeightedRandomSampler:
    ORGAN_WEIGHTS = {
        'lung':           3.0,   # raised from 1.5 — lung is hardest organ
        'spleen':         5.0,
        'kidney':         1.0,
        'prostate':       1.0,
        'largeintestine': 1.0,
    }
    weights = df['organ'].map(lambda o: ORGAN_WEIGHTS.get(o, 1.0)).values
    return WeightedRandomSampler(
        weights=torch.DoubleTensor(weights),
        num_samples=len(weights), replacement=True,
    )



## Model Architecture

In [ ]:


class FiLM(nn.Module):
    def __init__(self, cond_dim, num_channels):
        super().__init__()
        self.fc = nn.Linear(cond_dim, num_channels * 2)

    def forward(self, x, condition):
        gb = self.fc(condition).unsqueeze(-1).unsqueeze(-1)
        g, b = torch.chunk(gb, 2, dim=1)
        return x * (1 + g) + b


class MetadataEmbedder(nn.Module):
    def __init__(self, cond_dim=128):
        super().__init__()
        self.organ_emb  = nn.Embedding(6, 32)
        self.source_emb = nn.Embedding(3, 16)
        self.scale_mlp  = nn.Sequential(nn.Linear(1, 16), nn.SiLU())
        self.fusion     = nn.Sequential(
            nn.Linear(64, cond_dim), nn.SiLU(), nn.Linear(cond_dim, cond_dim))

    def forward(self, organ_idx, source_idx, pixel_scale):
        return self.fusion(torch.cat([
            self.organ_emb(organ_idx),
            self.source_emb(source_idx),
            self.scale_mlp(pixel_scale.view(-1, 1)),
        ], dim=1))


class ConvSiluFiLM(nn.Module):
    def __init__(self, in_ch, out_ch, cond_dim, ks=3):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, ks, padding=1)
        self.silu = nn.SiLU(inplace=True)
        self.film = FiLM(cond_dim, out_ch)

    def forward(self, x, c): return self.film(self.silu(self.conv(x)), c)


class MultimodalPathologyUNet(nn.Module):
    def __init__(self, encoder_name='convnex_Large', pretrained=False, cond_dim=128):
        super().__init__()
        self.encoder = timm.create_model(encoder_name, pretrained=pretrained,
                                         features_only=True)
        ef = [f['num_chs'] for f in self.encoder.feature_info]
        self.num_stages = len(ef)
        df = [32, 48, 64, 96, 128]
        self.metadata_embedder = MetadataEmbedder(cond_dim)
        bn = ef[-1]
        self.pred_organ  = nn.Linear(bn, 5)
        self.pred_source = nn.Linear(bn, 2)
        self.pred_scale  = nn.Linear(bn, 1)
        self.conv6   = ConvSiluFiLM(ef[-1],          df[-1], cond_dim)
        self.conv6_2 = ConvSiluFiLM(df[-1]+ef[-2],   df[-1], cond_dim)
        self.conv7   = ConvSiluFiLM(df[-1],           df[-2], cond_dim)
        self.conv7_2 = ConvSiluFiLM(df[-2]+ef[-3],   df[-2], cond_dim)
        self.conv8   = ConvSiluFiLM(df[-2],           df[-3], cond_dim)
        self.conv8_2 = ConvSiluFiLM(df[-3]+ef[-4],   df[-3], cond_dim)
        self.conv9   = ConvSiluFiLM(df[-3],           df[-4], cond_dim)
        self.conv9_2 = None if self.num_stages == 4 else \
                       ConvSiluFiLM(df[-4]+ef[-5], df[-4], cond_dim)
        self.conv10  = ConvSiluFiLM(df[-4], df[-5], cond_dim)
        self.res     = nn.Conv2d(df[-5], 1, 1)

    def forward(self, x, gt_organ=None, gt_source=None, gt_scale=None):
        feats = self.encoder(x)
        if self.num_stages == 4:
            e2, e3, e4, e5 = feats
        else:
            e1, e2, e3, e4, e5 = feats
        bn  = F.adaptive_avg_pool2d(e5, 1).view(x.shape[0], -1)
        po  = self.pred_organ(bn); ps = self.pred_source(bn); psc = self.pred_scale(bn)
        if gt_organ is not None:
            cond = self.metadata_embedder(gt_organ, gt_source, gt_scale)
        else:
            cond = self.metadata_embedder(torch.argmax(po, 1), torch.argmax(ps, 1), psc)
        up = lambda t: F.interpolate(t, scale_factor=2, mode='bilinear', align_corners=False)
        d6  = self.conv6(up(e5), cond)
        d6  = self.conv6_2(torch.cat([d6, e4], 1), cond)
        d7  = self.conv7(up(d6), cond)
        d7  = self.conv7_2(torch.cat([d7, e3], 1), cond)
        d8  = self.conv8(up(d7), cond)
        d8  = self.conv8_2(torch.cat([d8, e2], 1), cond)
        d9  = self.conv9(up(d8), cond)
        if self.num_stages == 5:
            d9 = self.conv9_2(torch.cat([d9, e1], 1), cond)
        out = self.res(self.conv10(d9, cond))
        return F.interpolate(out, scale_factor=2, mode='bilinear', align_corners=False)


class SegFormerGNDecoder(nn.Module):
    """
    MLP decoder for SegFormer with GroupNorm + FiLM conditioning.
    Mirrors the original SegformerDecodeHead but replaces BN with GN(32)
    and injects per-sample organ/source conditioning via FiLM.
    Outputs a single-channel logit map at 1/4 input resolution.
    """
    def __init__(self, in_channels: list[int], embed_dim: int = 256, cond_dim: int = 128):
        super().__init__()
        self.projections = nn.ModuleList([
            nn.Conv2d(c, embed_dim, kernel_size=1) for c in in_channels
        ])
        self.fuse_conv = nn.Sequential(
            nn.Conv2d(embed_dim * len(in_channels), embed_dim, kernel_size=1, bias=False),
            nn.GroupNorm(32, embed_dim),
            nn.ReLU(inplace=True),
        )
        self.film = FiLM(cond_dim, embed_dim)
        self.seg_head = nn.Conv2d(embed_dim, 1, kernel_size=1)

    def forward(self, features: list, cond: torch.Tensor) -> torch.Tensor:
        target_h, target_w = features[0].shape[2], features[0].shape[3]
        projected = []
        for proj, feat in zip(self.projections, features):
            x = proj(feat)
            if x.shape[2:] != (target_h, target_w):
                x = F.interpolate(x, size=(target_h, target_w),
                                  mode='bilinear', align_corners=False)
            projected.append(x)
        fused = self.fuse_conv(torch.cat(projected, dim=1))
        fused = self.film(fused, cond)
        return self.seg_head(fused)


class SegFormerPathologyModel(nn.Module):
    """
    SegFormer encoder (HuggingFace) + FiLM-conditioned GroupNorm MLP decoder.
    Drop-in replacement for MultimodalPathologyUNet — same forward signature,
    same single-channel output logit at input resolution.

    Key differences from the ConvNext UNet:
      - Transformer encoder (MixVisionTransformer mit-b4 by default)
      - GroupNorm(32) in decoder instead of BatchNorm (critical for batch_size=1)
      - Lower recommended LR: 6e-5 (vs 2e-4 for ConvNext)
    """
    def __init__(self, pretrained_name: str = 'nvidia/mit-b4', cond_dim: int = 128):
        super().__init__()
        cfg = SegformerConfig.from_pretrained(pretrained_name,
                                              output_hidden_states=True)
        self.encoder = SegformerModel.from_pretrained(pretrained_name, config=cfg)

        in_channels      = cfg.hidden_sizes          # e.g. [64, 128, 320, 512] for b4
        self.metadata_embedder = MetadataEmbedder(cond_dim)
        self.decoder     = SegFormerGNDecoder(in_channels, embed_dim=256, cond_dim=cond_dim)

        # Auxiliary heads for guided conditioning at inference
        bn_ch = in_channels[-1]
        self.pred_organ  = nn.Linear(bn_ch, 5)
        self.pred_source = nn.Linear(bn_ch, 2)
        self.pred_scale  = nn.Linear(bn_ch, 1)

    def forward(self, x, gt_organ=None, gt_source=None, gt_scale=None):
        outputs = self.encoder(pixel_values=x, output_hidden_states=True)
        # hidden_states: tuple of [B, C_i, H_i, W_i] for each stage
        hidden_states = outputs.hidden_states   # 4 stages for mit-b4

        # Global pool from deepest stage for conditioning
        bottleneck = F.adaptive_avg_pool2d(hidden_states[-1], 1).flatten(1)
        po  = self.pred_organ(bottleneck)
        ps  = self.pred_source(bottleneck)
        psc = self.pred_scale(bottleneck)

        if gt_organ is not None:
            cond = self.metadata_embedder(gt_organ, gt_source, gt_scale)
        else:
            cond = self.metadata_embedder(torch.argmax(po, 1), torch.argmax(ps, 1), psc)

        # Decode — output is at 1/4 input resolution, upsample to full
        logits_low = self.decoder(list(hidden_states), cond)
        logits     = F.interpolate(logits_low, size=x.shape[2:],
                                   mode='bilinear', align_corners=False)
        return logits


def build_model(use_segformer: bool = True, pretrained: bool = True) -> nn.Module:
    """Factory: returns the selected model architecture."""
    if use_segformer:
        print(f"  Building SegFormer ({SEGFORMER_VARIANT}) with GroupNorm decoder …")
        return SegFormerPathologyModel(pretrained_name=SEGFORMER_VARIANT)
    else:
        print("  Building ConvNext-Large UNet …")
        return MultimodalPathologyUNet('convnext_Large', pretrained=pretrained)



def _lovasz_grad(gt_sorted):
    """Lovász extension gradient."""
    n = gt_sorted.numel()
    gts = gt_sorted.sum()
    intersection = gts - gt_sorted.float().cumsum(0)
    union = gts + (1 - gt_sorted).float().cumsum(0)
    jaccard = 1. - intersection / union
    if n > 1:
        jaccard[1:] = jaccard[1:] - jaccard[:-1]
    return jaccard


def lovasz_hinge_flat(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    """Binary Lovász-hinge loss on flat tensors."""
    if labels.numel() == 0:
        return logits.sum() * 0.
    signs  = 2. * labels.float() - 1.
    errors = 1. - logits * signs
    errors_sorted, perm = torch.sort(errors, descending=True)
    gt_sorted = labels[perm]
    grad = _lovasz_grad(gt_sorted)
    return torch.dot(F.relu(errors_sorted), grad)


class CELovaszLoss(nn.Module):
    """
    CE:Lovasz = 1:3  (matches 1st place training settings).
    Works on single-channel logits with binary targets.
    """
    def __init__(self, ce_weight: float = 0.25, lovasz_weight: float = 0.75):
        super().__init__()
        self.ce_w  = ce_weight
        self.lov_w = lovasz_weight
        self.bce   = nn.BCEWithLogitsLoss()

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce_loss = self.bce(logits, targets)

        B = logits.shape[0]
        lov_loss = torch.stack([
            lovasz_hinge_flat(logits[i].view(-1), targets[i].view(-1).long())
            for i in range(B)
        ]).mean()

        return self.ce_w * ce_loss + self.lov_w * lov_loss


class FocalDiceLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, dice_weight=0.5, smooth=1e-5):
        super().__init__()
        self.alpha=alpha; self.gamma=gamma
        self.dice_weight=dice_weight; self.smooth=smooth
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, logits, targets):
        bce  = self.bce(logits, targets)
        prob = torch.sigmoid(logits)
        pt   = prob * targets + (1 - prob) * (1 - targets)
        fw   = (1 - pt) ** self.gamma
        aw   = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        fl   = (aw * fw * bce).mean()
        i    = (prob * targets).sum(dim=(2, 3))
        u    = prob.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
        dl   = 1.0 - ((2. * i + self.smooth) / (u + self.smooth)).mean()
        return (1 - self.dice_weight) * fl + self.dice_weight * dl




## Training

In [ ]:



def train_model():
    EPOCHS       = 30
    LR           = 2e-4
    # ── Resume from checkpoint ──────────────────────────────────────────────
    # Set RESUME_TRAINING=True (below, near SKIP_TRAINING) to load the best
    # saved checkpoint and continue training from where it left off.
    # The scheduler fast-forwards to the correct LR position automatically.
    resume_ckpt  = _ckpt_best  if RESUME_TRAINING and os.path.exists(_ckpt_best) else None
    resume_epoch = 0
    if resume_ckpt:
        _meta = json.load(open(_ckpt_info)) if os.path.exists(_ckpt_info) else {}
        resume_epoch = int(_meta.get('epoch', 0))
        print(f"\n  ↩ Resuming from epoch {resume_epoch} checkpoint  "
              f"(val_loss={_meta.get('val_loss','?'):.4f})")
        print(f"    Will train epochs {resume_epoch+1}–{EPOCHS} "
              f"({EPOCHS - resume_epoch} remaining)\n")

    # Scale batch size and accumulation to the number of available GPUs.
    n_gpus  = torch.cuda.device_count()
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    # Scale batch size to GPU VRAM so A100 (80GB) trains faster with larger batches
    # while T4 (16GB) stays safe. Effective batch always = 16.
    if vram_gb >= 70:        # A100 80GB
        BATCH_SIZE, ACCUM = 16, 1
    elif vram_gb >= 35:      # A100 40GB
        BATCH_SIZE, ACCUM = 8,  2
    elif vram_gb >= 20:      # L4 / A10
        BATCH_SIZE, ACCUM = 4,  4
    else:                    # T4 / V100 16GB
        BATCH_SIZE, ACCUM = 2,  8
    BATCH_SIZE = max(BATCH_SIZE, 2 * n_gpus)  # multi-GPU floor
    print(f"[INFO] GPUs={n_gpus}  BATCH_SIZE={BATCH_SIZE}  ACCUM={ACCUM}  "
          f"effective_batch={BATCH_SIZE * ACCUM}")

    # Fit stain normalizer on HuBMAP reference image before any data loading
    _build_stain_normalizer(HUBMAP_IMG_DIR)

    # HuBMAP train/val split — stratified by organ
    hubmap_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
    h_train, h_val = train_test_split(
        hubmap_df, test_size=0.2, stratify=hubmap_df['organ'], random_state=42)
    print(f"[INFO] HuBMAP train={len(h_train)}  val={len(h_val)}")

    ext_df = build_external_df(lung_manifest, spleen_manifest, pseudo_hubmap_rows)

    if not ext_df.empty:
        ext_with_mask  = ext_df[ext_df['has_real_mask']].reset_index(drop=True)
        ext_train_only = ext_df[~ext_df['has_real_mask']].reset_index(drop=True)
        print(f"[INFO] External with mask={len(ext_with_mask)}  train-only={len(ext_train_only)}")

        if len(ext_with_mask) >= 5:
            try:
                e_train_masked, e_val = train_test_split(
                    ext_with_mask, test_size=0.2,
                    stratify=ext_with_mask['organ'], random_state=42)
            except ValueError:
                # fallback when a class has < 2 samples after split
                e_train_masked = ext_with_mask.sample(frac=0.8, random_state=42)
                e_val = ext_with_mask.drop(e_train_masked.index)
        else:
            e_train_masked = ext_with_mask
            e_val          = pd.DataFrame()

        e_train = e_train_masked
        print(f"[INFO] External train rows (masked only): {len(e_train)}  "
              f"  (excluded {len(ext_train_only)} maskless rows)")
    else:
        e_train = pd.DataFrame()
        e_val   = pd.DataFrame()

    full_train = pd.concat([h_train, e_train], ignore_index=True)
    full_val   = pd.concat([h_val,   e_val  ], ignore_index=True)
    print(f"[INFO] Total train={len(full_train)}  val={len(full_val)}")

    train_ds = CombinedTrainDataset(full_train, HUBMAP_IMG_DIR, IMG_SIZE, is_train=True)
    val_ds   = CombinedTrainDataset(full_val,   HUBMAP_IMG_DIR, IMG_SIZE, is_train=False)

    sampler = make_weighted_sampler(full_train)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                              num_workers=2, drop_last=True, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=True)
    print(f"[INFO] Train batches: {len(train_loader)}  Val batches: {len(val_loader)}\n")

    model = build_model(use_segformer=USE_SEGFORMER, pretrained=True).cuda()
    if resume_ckpt:
        _sd = torch.load(resume_ckpt, map_location='cuda')
        model.load_state_dict(_sd['state_dict'])
        print(f"  ✓ Weights loaded from checkpoint")
    if n_gpus > 1:
        print(f"  Wrapping model in DataParallel across {n_gpus} GPUs")
        model = nn.DataParallel(model)

    criterion    = (CELovaszLoss() if USE_SEGFORMER else FocalDiceLoss()).cuda()
    effective_lr = 6e-5 if USE_SEGFORMER else LR
    optimizer    = torch.optim.AdamW(model.parameters(), lr=effective_lr, weight_decay=1e-4)
    total_steps  = len(train_loader) // ACCUM * EPOCHS
    scheduler    = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=total_steps, eta_min=1e-6)
    if resume_epoch > 0:
        steps_done = len(train_loader) // ACCUM * resume_epoch
        for _ in range(steps_done):
            scheduler.step()
        print(f"  ✓ Scheduler fast-forwarded {steps_done} steps  "
              f"LR={optimizer.param_groups[0]['lr']:.2e}")

    scaler    = torch.amp.GradScaler('cuda')
    best_val  = float('inf')
    dashboard = TrainingDashboard(EPOCHS, len(train_loader), len(val_loader))

    n_remaining = EPOCHS - resume_epoch
    print(f"Training epochs {resume_epoch+1}–{EPOCHS}  "
          f"({n_remaining} remaining) — dashboard updates after each val pass.\n")

    for epoch in range(resume_epoch, EPOCHS):
        model.train(); train_loss = 0.0; optimizer.zero_grad()
        pbar = tqdm(train_loader, desc=f"Ep {epoch+1:>2}/{EPOCHS} [train]",
                    unit="batch", dynamic_ncols=True)
        for step, batch in enumerate(pbar):
            imgs    = batch['img'].cuda()
            masks   = batch['mask'].cuda()
            organs  = batch['organ_idx'].cuda()
            sources = batch['source_idx'].cuda()
            scales  = batch['pixel_scale'].float().cuda()
            with torch.amp.autocast('cuda'):
                logits = model(imgs, gt_organ=organs, gt_source=sources, gt_scale=scales)
                loss   = criterion(logits, masks) / ACCUM
            scaler.scale(loss).backward()
            if (step + 1) % ACCUM == 0 or (step + 1) == len(train_loader):
                scaler.step(optimizer); scaler.update()
                optimizer.zero_grad(); scheduler.step()
            cl = loss.item() * ACCUM; train_loss += cl
            pbar.set_postfix(loss=f"{cl:.4f}",
                             lr=f"{optimizer.param_groups[0]['lr']:.2e}",
                             best=f"{best_val:.4f}" if best_val < float('inf') else "—")
        avg_train = train_loss / len(train_loader)

        model.eval(); val_loss = 0.0; organ_dice = {o: [] for o in ORGAN_MAP}
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Ep {epoch+1:>2}/{EPOCHS} [val]  ",
                              unit="batch", dynamic_ncols=True):
                imgs    = batch['img'].cuda(); masks = batch['mask'].cuda()
                organs  = batch['organ_idx'].cuda()
                sources = batch['source_idx'].cuda()
                scales  = batch['pixel_scale'].float().cuda()
                on_list = batch['organ_name']; sn_list = batch['source_name']
                with torch.amp.autocast('cuda'):
                    logits = model(imgs, gt_organ=organs, gt_source=sources, gt_scale=scales)
                    val_loss += criterion(logits, masks).item()
                probs = torch.sigmoid(logits)
                for i in range(len(imgs)):
                    on     = on_list[i]; sn = sn_list[i]
                    thresh = ORGAN_THRESHOLDS.get(sn, ORGAN_THRESHOLDS['HPA']).get(on, 127) / 255.0
                    pred   = (probs[i] > thresh).float()
                    inter  = (pred * masks[i]).sum(); union = pred.sum() + masks[i].sum()
                    if on in organ_dice:
                        organ_dice[on].append(((2. * inter + 1e-5) / (union + 1e-5)).item())

        avg_val  = val_loss / len(val_loader); cur_lr = optimizer.param_groups[0]['lr']
        per_dice = {o: float(np.mean(s)) for o, s in organ_dice.items() if s}
        if avg_val < best_val:
            best_val = avg_val
            ckpt_path = os.path.join(MODELS_DIR, 'multimodal_unet_best.pth')
            # ── Pre-save: verify Drive has enough space, auto-free if needed ──
            def _ensure_drive_space(required_mb=500):
                if not _DRIVE_AVAILABLE:
                    return True
                try:
                    import shutil as _shutil
                    _, _, free = _shutil.disk_usage('/content/drive')
                    free_mb = free / 1e6
                    if free_mb >= required_mb:
                        return True
                    _z = os.path.join(EXT_DIR, 'lung_hpa', 'zenodo_team2')
                    _sentinel = os.path.join(EXT_DIR, 'lung_hpa', '.zenodo_done')
                    if os.path.exists(_z):
                        import shutil as _sh; _sh.rmtree(_z)
                        # Re-touch sentinel outside the deleted dir
                        pathlib.Path(_sentinel).touch()
                        _, _, free = _shutil.disk_usage('/content/drive')
                        free_mb = free / 1e6
                        print(f"\n  ⚠ Drive was full — deleted Zenodo image cache to free space. "
                              f"Free now: {free_mb:.0f} MB (sentinel preserved — won't re-download)")
                    if free_mb < required_mb:
                        print(f"\n  ✗ Drive still has only {free_mb:.0f} MB free — "
                              f"checkpoint NOT saved. Free up Drive space to fix this.")
                        return False
                    return True
                except Exception:
                    return True  # non-Drive path, always ok
            # Unwrap DataParallel before saving so the checkpoint loads on any GPU count
            _m = model.module if isinstance(model, nn.DataParallel) else model
            if _ensure_drive_space(required_mb=600):
                torch.save({'epoch': epoch + 1, 'state_dict': _m.state_dict(),
                            'val_loss': avg_val, 'organ_dice': per_dice}, ckpt_path)
                # Human-readable sidecar so you can always find the best checkpoint
                info = {'best_model_path': ckpt_path, 'epoch': epoch + 1,
                        'val_loss': float(avg_val), 'organ_dice': per_dice}
                with open(os.path.join(MODELS_DIR, 'best_model_info.json'), 'w') as _f:
                    json.dump(info, _f, indent=2)
                print(f"\n  ✓ NEW BEST  epoch={epoch+1}  val_loss={avg_val:.4f}"
                      f"  → {ckpt_path}")
        dashboard.update(epoch, avg_train, avg_val, organ_dice, cur_lr)



## Inference and Submission

In [ ]:
def run_inference():
    # Read the best model path from the sidecar written during training.
    # Falls back to the default filename if the sidecar is missing.
    sidecar = os.path.join(MODELS_DIR, 'best_model_info.json')
    if os.path.exists(sidecar):
        with open(sidecar) as _f:
            _info = json.load(_f)
        ckpt_path = _info['best_model_path']
        print(f"Loading best model from sidecar: {ckpt_path}  "
              f"(epoch {_info['epoch']}, val_loss {_info['val_loss']:.4f})")
        print(f"  Organ Dice at save: {_info.get('organ_dice', {})}")
    else:
        ckpt_path = os.path.join(MODELS_DIR, 'multimodal_unet_best.pth')
        print(f"No sidecar found — loading {ckpt_path}")

    df      = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
    dataset = HubmapTestDataset(df, os.path.join(DATA_DIR, 'test_images'), IMG_SIZE)
    loader  = DataLoader(dataset, batch_size=TEST_BATCH_SIZE, num_workers=2, shuffle=False)
    model   = build_model(use_segformer=USE_SEGFORMER, pretrained=False)
    ckpt    = torch.load(ckpt_path, map_location='cpu')
    model.load_state_dict(ckpt['state_dict']); model = model.eval().cuda()
    results = []
    with torch.no_grad():
        for sample in tqdm(loader, desc="Predicting"):
            img_id = sample['id'][0]
            orig_h = sample['orig_h'].item(); orig_w = sample['orig_w'].item()
            oi = sample['organ_idx'].cuda(); si = sample['source_idx'].cuda()
            ps = sample['pixel_size'].float().cuda()
            inp = sample['img'].numpy()
            acc = np.zeros((orig_h, orig_w), dtype='float32')
            with torch.amp.autocast('cuda'):
                for t in range(4):
                    flip = t % 2 == 1; rot = t // 2
                    x = inp.copy()
                    if rot:  x = np.rot90(x, k=rot,   axes=(2, 3)).copy()
                    if flip: x = x[:, :, :, ::-1].copy()
                    out = model(torch.from_numpy(x).cuda(), gt_organ=oi, gt_source=si, gt_scale=ps)
                    p   = torch.sigmoid(out).float().cpu().numpy()[0, 0]
                    if flip: p = p[:, ::-1].copy()
                    if rot:  p = np.rot90(p, k=4 - rot, axes=(0, 1)).copy()
                    acc += cv2.resize(p, (orig_w, orig_h))
            acc /= 4.0
            on     = sample['organ_name'][0]; sn = sample['source_name'][0]
            thresh = ORGAN_THRESHOLDS[sn][on] / 255.0
            results.append({'id': img_id,
                            'rle': rle_encode_less_memory((acc > thresh).astype(np.uint8))})
    pd.DataFrame(results).to_csv(SUBMISSION_FILE, index=False)
    print(f"Submission saved. {(timeit.default_timer() - t0) / 60:.1f} min elapsed.")

if SKIP_TRAINING:
    print("\n── SKIP_TRAINING=True — skipping to inference using saved checkpoint ──")
    if not os.path.exists(_ckpt_best):
        raise FileNotFoundError(
            f"SKIP_TRAINING=True but no checkpoint found at {_ckpt_best}\n"
            f"Set SKIP_TRAINING=False to train first."
        )
elif RESUME_TRAINING:
    if not os.path.exists(_ckpt_best):
        raise FileNotFoundError(
            f"RESUME_TRAINING=True but no checkpoint found at {_ckpt_best}\n"
            f"Set RESUME_TRAINING=False to train from scratch."
        )
    print("\n── Resuming training from saved checkpoint ──")
    train_model()
else:
    print("\n── Starting training ──")
    train_model()

print("\n── Running inference ──")
run_inference()
